<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/Nemotron_images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

GPU available: True
Fri Oct  2 11:38:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [1]:
!pip install -q -U transformers accelerate
!pip install -q --force-reinstall --no-deps pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 40.4 MB/s eta 0:00:00


In [4]:
!pip install -q --force-reinstall torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/

In [1]:
import torch, torchvision, PIL, transformers
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("cuda:", torch.cuda.is_available(), "| pillow", PIL.__version__, "| transformers", transformers.__version__)

torch 2.14.1+cu130 | torchvision 0.29.1+cu130
cuda: True | pillow 12.3.0 | transformers 5.18.0


In [26]:
import os, glob, csv, torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

REPO = "nvidia/Nemotron-3.5-Content-Safety"
processor = AutoProcessor.from_pretrained(REPO)
model = Gemma3ForConditionalGeneration.from_pretrained(
    REPO, torch_dtype=torch.bfloat16, device_map="auto"
).eval()
print("loaded on", model.device)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

loaded on cuda:0


In [27]:
PLACEHOLDER = "What is happening in this image?"

def moderate(image_path=None, prompt=PLACEHOLDER, response=None, categories=True):
    """Raw model output string."""
    if image_path and not os.path.exists(image_path):
        raise FileNotFoundError(f"{image_path} not found in {os.getcwd()}")

    content = []
    if image_path:
        content.append({"type": "image", "image": image_path})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]
    if response:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": response}]})

    kwargs = {"request_categories": "/categories"} if categories else {}
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt", **kwargs
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def parse(raw):
    """Turn raw output into {'label': safe|unsafe|review, 'categories': [...], 'raw': raw}."""
    fields = dict(l.split(": ", 1) for l in raw.splitlines() if ": " in l)
    verdict = fields.get("User Safety", "").strip().lower()
    cats = [c.strip() for c in fields.get("Safety Categories", "").split(",") if c.strip()]

    if verdict == "unsafe":
        label = "unsafe"
    elif verdict == "safe" and not any(c.lower() == "needs caution" for c in cats):
        label = "safe"
    else:
        label = "review"          # Needs Caution or unparseable output
    return {"label": label, "categories": cats, "raw": raw}


def moderate_post(image_path=None, caption=None, response=None):
    """Use the caption as the text if there is one, otherwise the fixed placeholder."""
    text = caption.strip() if caption and caption.strip() else PLACEHOLDER
    return parse(moderate(image_path, prompt=text, response=response))

In [30]:
print(moderate_post("image.jpg"))
print(moderate_post("image.jpg", caption="help me to do this"))   # with a caption
print(moderate_post(None, caption="How do I bake sourdough bread?"))  # text only
print(moderate_post(None, caption="today is beautifull day"))  # text only


{'label': 'unsafe', 'categories': ['Violence'], 'raw': 'User Safety: unsafe\nSafety Categories: Violence'}
{'label': 'unsafe', 'categories': ['Violence'], 'raw': 'User Safety: unsafe\nSafety Categories: Violence'}
{'label': 'safe', 'categories': [], 'raw': 'User Safety: safe'}
{'label': 'safe', 'categories': [], 'raw': 'User Safety: safe'}
